In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.chdir('/content/drive/MyDrive/AI_Project/DLA')

In [ ]:
datasets = [
  {
      "name": "CROHME HME train",
      "path": "data/preprocessed/train/paired/crohme/hme_preprocessed",
      "image_ext":"png"
  },
  {
      "name": "CROHME PME train",
      "path": "data/preprocessed/train/paired/crohme/pme_preprocessed",
      "image_ext":"png"
  },
  {
      "name": "IM2LATEX HME train",
      "path": "data/preprocessed/train/paired/im2latex/hme",
      "image_ext":"png"
  },
  {
      "name": "IM2LATEX PME train",
      "path": "data/preprocessed/train/paired/im2latex/pme_cropped",
      "image_ext":"png"
  },
  {
      "name": "CROHME 2014 PME test",
      "path": "data/preprocessed/test/pme/crohme/2014/pme_img",
      "image_ext":"png"
  },
  {
      "name": "CROHME 2016 PME test",
      "path": "data/preprocessed/test/pme/crohme/2016/pme_img",
      "image_ext":"png"
  },
  {
      "name": "CROHME 2019 PME test",
      "path": "data/preprocessed/test/pme/crohme/2019/pme_img",
      "image_ext":"png"
  },
  {
      "name": "CROHME 2014 HME test",
      "path": "data/preprocessed/test/hme/crohme/2014/hme_img_preprocessed",
      "image_ext":"png"
  },
  {
      "name": "CROHME 2016 HME test",
      "path": "data/preprocessed/test/hme/crohme/2016/hme_img_preprocessed",
      "image_ext":"png"
  },
  {
      "name": "CROHME 2019 HME test",
      "path": "data/preprocessed/test/hme/crohme/2019/hme_img_preprocessed",
      "image_ext":"png"
  },
    {
      "name": "IM2LATEX PME test",
      "path": "data/preprocessed/test/pme/im2latex/img",
      "image_ext":"png"
  },
    {
      "name": "IM2LATEX HME test",
      "path": "data/preprocessed/test/pme/im2latex/img",
      "image_ext":"png"
  }
]

In [ ]:
!pip install rapidfuzz

# Data utils

## 최소한의 성능 확인 위해 sub-optimal하게 구현

- 추후 고려해 볼 내용 (ablation study)
  - transform 함수를 미니배치단위로 동적으로 사이즈 맞추게끔해서 메모리 절약
  - 학습 시 정규화를 densenet pretrain이 아닌, 현재 데이터에 맞춰서 정규화 진행
  - densenet pretrain weight 사용 o,x 경우 비교
  - densenet pretrain 처음 채널을 1채널 (gray)로 할지 말지 유무에 따른 차이 비교
  - scale augmentation 이용

In [ ]:
from pathlib import Path
from torch.utils.data import Dataset
from PIL import Image
import torch

# unpaired 이미지, 캡션 반환
class UnpairedDataset(Dataset):
    def __init__(self, image_dir, caption_path, transform=None, image_ext="png", vocab=None):
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.image_ext = image_ext
        self.vocab = vocab
        self.data = []

        # 초기 로딩 시 필터링 (토큰 길이 <2 건너뛰기)
        with open(caption_path, "r", encoding="utf-8") as f:
          for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 2:
                continue
            image_id, latex = parts
            image_name = f"{image_id}.{self.image_ext}" if '.' not in image_id else image_id
            image_path = self.image_dir / image_name
            if image_path.exists():
                self.data.append((image_path, latex))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path, latex = self.data[idx]
        image = Image.open(image_path).convert('L')

        if self.transform:
            image = self.transform(image)

        formula_ids = self.vocab.encode(latex)
        # 길이 2 이하인 경우 None 반환
        if len(formula_ids) <= 2:
            return None
        formula_tensor = torch.tensor(formula_ids, dtype=torch.long)
        return {"image": image, "formula": formula_tensor}

# paired 이미지, 캡션 반환
class PairedDataset(Dataset):
    def __init__(self, hme_dir, pme_dir, caption_path, transform_hme, transform_pme, image_exts=["png", "png"], vocab=None):
        self.hme_dir = Path(hme_dir)
        self.pme_dir = Path(pme_dir)
        self.transform_hme = transform_hme
        self.transform_pme = transform_pme
        self.img_exts = image_exts
        self.vocab = vocab
        self.data = []

        # 초기 로딩 시 필터링 (토큰 길이 <2 건너뛰기)
        with open(caption_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) != 2:
                    continue
                image_id, latex = parts
                self.data.append((image_id, latex))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_id, formula = self.data[idx]

        img_hme_name = f"{image_id}.{self.image_exts[0]}" if '.' not in image_id else image_id
        img_pme_name = f"{image_id}.{self.image_exts[1]}" if '.' not in image_id else image_id

        img_hme_path = self.hme_dir / img_hme_name
        img_pme_path = self.pme_dir / img_pme_name

        if img_hme_path.exists():
            img_hme = Image.open(img_hme_path).convert('L')
        if img_pme_path.exists():
            img_pme = Image.open(img_pme_path).convert('L')

        if self.transform_hme:
            img_hme = self.transform_hme(img_hme)
        if self.transform_pme:
            img_pme = self.transform_pme(img_pme)

        formula_ids = self.vocab.encode(formula)
        # 길이 2 이하인 경우 None 반환
        if len(formula_ids) <= 2:
            return None
        formula_tensor = torch.tensor(formula_ids, dtype=torch.long)
        return {"img_hme": img_hme, "img_pme": img_pme, "formula": formula_tensor}

In [ ]:
import torch
import torch.nn.functional as F

# unpaired collate (pad_value=1.0 -> 흰색, 0 -> 회색, (0-mean)/std -> 검정색)
def unpaired_collate_fn(batch, pad_idx=0, return_mask=True, pad_value=1.0):
    batch = [b for b in batch if b is not None]

    # 배치 내 최대 폭으로 오른쪽 제로 패딩
    imgs = [b["image"] for b in batch]   # [1, 128, W_i]
    widths = [x.shape[-1] for x in imgs]
    Wmax = max(widths)

    # 패딩 진행
    padded_imgs, masks = [], []
    for img, w in zip(imgs, widths):
        if w < Wmax:
            img = F.pad(img, (0, Wmax - w, 0, 0), value=pad_value)   # (w_left, w_right, h_top, h_bottom)
        padded_imgs.append(img)
        if return_mask:
            m = torch.zeros(Wmax, dtype=torch.bool)
            m[:w] = True
            masks.append(m)

    # 스택
    images = torch.stack(padded_imgs, dim=0)   # [B, 1, 128, Wmax]
    formulas = torch.nn.utils.rnn.pad_sequence(
        [b["formula"] for b in batch], batch_first=True, padding_value=pad_idx
    )   # [B, Lmax]
    out = {"image": images, "formula": formulas}

    if return_mask:
        out["image_mask"] = torch.stack(masks, dim=0)   # [B, Wmax] (어텐션 마스크에 사용)
    return out

# paired collate (pad_value=1.0 -> 흰색, 0 -> 회색, (0-mean)/std -> 검정색)
def paired_collate_fn(batch, pad_idx=0, return_mask=True, pad_value=1.0):
    batch = [b for b in batch if b is not None]

    # 각 도메인별 배치 내 최대 폭으로 오른쪽 제로 패딩
    imgs_hme = [b["img_hme"] for b in batch]   # [1, H, W_h_i]
    imgs_pme = [b["img_pme"] for b in batch]   # [1, H, W_p_i]
    widths_h = [x.shape[-1] for x in imgs_hme]
    widths_p = [x.shape[-1] for x in imgs_pme]
    Wmax_h = max(widths_h)
    Wmax_p = max(widths_p)

    # hme 패딩 진행
    padded_hme, masks_h = [], []
    for img, w in zip(imgs_hme, widths_h):
          if w < Wmax_h:
              img = F.pad(img, (0, Wmax_h - w, 0, 0), value=pad_value)
          padded_hme.append(img)
          if return_mask:
              m = torch.zeros(Wmax_h, dtype=torch.bool)
              m[:w] = True
              masks_h.append(m)

    # pme 패딩 진행
    padded_pme, masks_p = [], []
    for img, w in zip(imgs_pme, widths_p):
        if w < Wmax_p:
            img = F.pad(img, (0, Wmax_p - w, 0, 0), value=pad_value)
        padded_pme.append(img)
        if return_mask:
            m = torch.zeros(Wmax_p, dtype=torch.bool)
            m[:w] = True
            masks_p.append(m)

    # 스택
    imgs_hme = torch.stack(padded_hme, dim=0)  # [B, 1, H, Wmax_h]
    imgs_pme = torch.stack(padded_pme, dim=0)  # [B, 1, H, Wmax_p]
    formulas = torch.nn.utils.rnn.pad_sequence(
        [b["formula"] for b in batch], batch_first=True, padding_value=pad_idx
    )   # [B, Lmax]
    out = {"img_hme": imgs_hme, "img_pme": imgs_pme, "formula": formulas}

    if return_mask:
        out["image_mask_hme"] = torch.stack(masks_h, dim=0)  # [B, Wmax_h]
        out["image_mask_pme"] = torch.stack(masks_p, dim=0)  # [B, Wmax_p]
    return out

## Vocab

In [ ]:
from pathlib import Path
from typing import List
import re

# 숫자+단위 분리 (나머지는 공백 기준 토큰 유지)
def tokenize_formula(formula):
    ans = []
    tokens = formula.strip().split()
    unit_pat = re.compile(r'^([+-]?\d+(?:\.\d+)?)(cm|mm|pt|in|ex|em)$')
    for tok in tokens:
        m = unit_pat.match(tok)
        if m:
            num, unit = m.groups()
            ans.append(num)
            ans.append(unit)
        else:
            ans.append(tok)
    return ans

class Vocab:
    PAD_TOKEN = "<pad>"
    SOS_TOKEN = "<sos>"
    EOS_TOKEN = "<eos>"
    UNK_TOKEN = "<unk>"

    def __init__(self):
        self.tokens = [self.PAD_TOKEN, self.SOS_TOKEN, self.EOS_TOKEN, self.UNK_TOKEN]
        self.token2idx = {tok: idx for idx, tok in enumerate(self.tokens)}
        self.idx2token = self.tokens.copy()

'''
vocab 있는 걸로 사용해서 새로 안만들어도 됨
'''

    # def build_vocab(self, formula_list: List[str]):
    #     for formula in formula_list:
    #         for tok in tokenize_formula(formula):
    #             if tok not in self.token2idx:
    #                 self.token2idx[tok] = len(self.idx2token)
    #                 self.idx2token.append(tok)

    # 특수 토큰 추가
    def encode(self, formula: str) -> List[int]:
        tokens = tokenize_formula(formula)
        sos = self.token2idx[self.SOS_TOKEN]
        eos = self.token2idx[self.EOS_TOKEN]
        unk = self.token2idx[self.UNK_TOKEN]
        ids = [sos]
        ids.extend(self.token2idx.get(t, unk) for t in tokens)  # OOV -> <unk>
        ids.append(eos)
        return ids

    # 특수 토큰 제거
    def decode(self, token_ids: List[int]) -> str:
        out: List[str] = []
        for idx in token_ids:
            if not (0 <= idx < len(self.idx2token)):
                continue
            tok = self.idx2token[idx]
            if tok == self.EOS_TOKEN:   # EOS에서 중단
                break
            if tok in (self.PAD_TOKEN, self.SOS_TOKEN):   # 특수토큰 제거
                continue
            out.append(tok)
        return " ".join(out)

    def __len__(self):
        return len(self.idx2token)

    def save_to_txt(self, path: Path):
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            for token in self.idx2token:
                f.write(token + '\n')

    @classmethod
    def load_from_txt(cls, path: Path) -> "Vocab":
        vocab = cls()
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                token = line.strip()
                if token not in vocab.token2idx:
                    vocab.token2idx[token] = len(vocab.idx2token)
                    vocab.idx2token.append(token)
        return vocab

def load_caption_formulas(caption_path: Path) -> List[str]:
    formulas = []
    with open(caption_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            _, formula = parts
            formulas.append(formula)
    return formulas

# Encoder

### 변경사항
- 기존 2D attention 대신, 계산 및 메모리 효율, 구현 용이성을 위해 1D attention으로 변경
  - 이때, 어텐션을 위해서는 시퀀스형태로 데이터를 받아와줘야하므로 (B,C,H,W)인 이미지 데이터를 어떻게 (B,T,D) 형태로 변경할지 고민 필요
  - ViT가 아닌 트랜스포머 이전 CNN기반 인코더들에서는 어떻게 이런형태 만들지? Region Proposal하거나 단순히 grid로 쪼개서 사용하나? grid로 쪼갤 경우 위치 정보는 어떻게 부여하지?
  - 일단 코드에는 (B,C,H',W') feature를 H'축에서 평균해서 (B,C,1,W')로 만들고 (B,W',C) 형태로 사용하도록 함, 추후 (B, H'W', C) 등 다른방식 고려해볼 수 있을 것

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import densenet121, DenseNet121_Weights

class Encoder(nn.Module):
    """
    Input : x -> (B, 1, H=128, W)
    Output : seq -> (B, C=1024, H'=4, W')
    """
    def __init__(self, pretrained: bool=True):
        super().__init__()

        # 사전학습 가중치 선택 (A/B 테스트 돌려보면 좋을 듯?)
        w = DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        densenet = densenet121(weights=w)

        # conv0 가중치만 미리 저장 -> 1채널 conv 교체 후 복사
        old_w = densenet.features.conv0.weight.detach().clone()
        densenet.features.conv0 = nn.Conv2d(
            in_channels=1, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False
        )

        if pretrained:
            with torch.no_grad():
                # gpu 메모리 소모 줄이기 위해 그레이스케일 변환
                densenet.features.conv0.weight.copy_(old_w.mean(dim=1, keepdim=True))

        # norm5까지 구조 유지
        self.backbone = densenet.features

        # 사전학습 가중치 사용안하면 초기화
        if not pretrained:
            self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # 2D feature map 추출
        x = self.backbone(x)                   # (B, 1, H=128, W) -> (B, C=1024, Hp=H/32, Wp=W/32)
        x = F.relu(x, inplace=False)           # (B, 1024 ,4, Wp)
        return x                               # (B, 1024, 4, Wp)

- Encoder 차원 동작 테스트

In [ ]:
import os
import cv2
import numpy as np
import torch

# Data utils 대신 바로 텐서화
def to_tensor(img):
    if img.ndim == 3:  # 컬러면 그레이스케일로 변환
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = img.astype(np.float32) / 255.0
    t = torch.from_numpy(img)[None, None, ...]
    return t

def test_encoder_once(encoder, image_path, device=None):
    assert os.path.exists(image_path), f"이미지 경로 에러: {image_path}"
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    encoder = encoder.to(device).eval()

    # 이미지 로드
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise RuntimeError("이미지 로드 에러")
    x = to_tensor(img).to(device)

    with torch.no_grad():
        y = encoder(x)  # (B,C,H'W')

    # 결과 출력
    print(f"Original image shape : {img.shape}")               # (H, W)
    print(f"Input tensor shape   : {tuple(x.shape)}")          # (1, 1, H, W)
    print(f"Encoder output shape : {tuple(y.shape)}")          # (1, C, H', W')

In [ ]:
IMAGE_PATH = "dummy/preprocessed/train/paired/crohme/hme_preprocessed/101_Fabricio.png"
enc = Encoder(pretrained=True)
test_encoder_once(enc, IMAGE_PATH)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 159MB/s]


Original image shape : (97, 412)
Input tensor shape   : (1, 1, 97, 412)
Encoder output shape : (1, 1024, 3, 12)


# Decoder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ProjectedDotAttention(nn.Module):
    """
    query  : (B, q_dim)            # 디코더 hidden
    memory : (B, T_enc, k_dim)     # 인코더 시퀀스 (to_seq 이후)
    mask   : (B, T_enc) bool or None  True=유효
    returns:
      context : (B, k_dim)
      attn    : (B, T_enc)
    """
    def __init__(self, q_dim: int, k_dim: int, attn_dim: int = 256):
        super().__init__()
        self.q_proj = nn.Linear(q_dim, attn_dim, bias=False)
        self.k_proj = nn.Linear(k_dim, attn_dim, bias=False)
        self.scale = attn_dim ** 0.5

    def forward(self, query, memory, mask=None):
        # Q: (B,1,A), K: (B,T,A)
        Q = self.q_proj(query).unsqueeze(1)
        K = self.k_proj(memory)

        # 점수: (B,T)
        scores = torch.bmm(Q, K.transpose(1, 2)).squeeze(1) / self.scale
        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))

        attn = F.softmax(scores, dim=-1)                # (B,T)
        context = torch.bmm(attn.unsqueeze(1), memory)  # (B,1,k_dim)
        context = context.squeeze(1)                    # (B,k_dim)
        return context, attn

In [ ]:
import torch
import torch.nn as nn

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, hidden_dim=512,
                  enc_dim=1024, attn_dim=512, num_layers=1):
        """
        embed: 토큰 id -> 임베딩 벡터 (id 0=패딩 고정)
        gru  : (B, 1, emb_dim) -> 출력 (B, 1, H), hidden (L, B, H)
        attn : 쿼리(B, H), 인코더 메모리(B, T_enc, D_enc) -> attn_dim
        mix  : 디코더 state + context -> 다음에 사용할 fused sate
        fc   : logit 출력
        """
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.attn = ProjectedDotAttention(hidden_dim, enc_dim, attn_dim)
        self.mix = nn.Linear(hidden_dim + enc_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self._init()

    def _init(self):
        nn.init.uniform_(self.embed.weight, -0.1, 0.1)
        for name, param in self.gru.named_parameters():
            # input -> hidden 가중치
            if 'weight_ih' in name:
              nn.init.xavier_uniform_(param)
            # hidden -> hidden 가중치
            elif 'weight_hh' in name:
              nn.init.orthogonal_(param)
            elif 'bias' in name:
              nn.init.zeros_(param)

        for module in [self.mix, self.fc]:
          nn.init.xavier_uniform_(module.weight)
          nn.init.zeros_(module.bias)

    def forward(self, prev_token, hidden, enc_memory, enc_mask=None):
        """
        prev_token: (B,) -> teacher forcing에서 정답 이전 토큰, 추론에서 이전 스텝 토큰 예측
        hidden    : (L, B, H) -> GRU hidden
        enc_memory: (B, T_enc, D_enc) -> 인코더 시퀀스 메모리
        enc_mask  : (B, T_enc) True=keep -> attn에서 패딩 위치 마스킹
        returns:
          logits        : (B, V)
          hidden        : (L, B, H)
          context_vector: (B, enc_dim)
          attn_weights  : (B, T_enc)
        """
        embedded = self.embed(prev_token).unsqueeze(1)  # (B, 1, emb_dim)
        output, hidden = self.gru(embedded, hidden)     # (B, 1, H)
        decoder_state = output.squeeze(1)               # (B, H)
        context_vector, attn_weights = self.attn(decoder_state, enc_memory, enc_mask)   # (B, A), (B, T_enc)
        fused_state = torch.tanh(self.mix(torch.cat([decoder_state, context_vector], dim=-1)))   #(B, H)
        output_logits = self.fc(output.squeeze(1))  # (B, vocab_size)

        return output_logits, hidden, context_vector, attn_weights

# Attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Attention(nn.Module):
    """
    Args:
        q_dim   : 디코더 hidden 차원
        ctx_dim : 인코더 2D feature 채널 수 C=1024
        attn_dim: 내부 어텐션 hidden 차원
    forward(query, F2d, cov=None, mask=None)
        query   : (B, q_dim) -> 현재 스텝 h_t
        F2d     : (B, C, H', W') -> 인코더 2D feature, key/value 역할
        cov     : (B, 1, H', W') or None -> 직전까지의 누적 coverage map
        mask    : (B, H', W') bool -> True=유효, False=패딩
    returns:
        context : (B, C)
        alpha   : (B, H', W')
        cov_next: (B, 1, H', W')
    """
    def __init__(self, q_dim: int, ctx_dim: int, attn_dim: int=256):
        super().__init__()
        self.W_F   = nn.Conv2d(ctx_dim, attn_dim, kernel_size=1, bias=False)  # keys
        self.W_cov = nn.Conv2d(1, attn_dim, kernel_size=1, bias=False)        # coverage (location-aware)
        self.W_h   = nn.Linear(q_dim, attn_dim, bias=False)                   # query
        self.v_att = nn.Parameter(torch.randn(attn_dim))                      # projection

    @torch.no_grad()
    def init_coverage(self, B: int, Hp: int, Wp: int, device=None):
        return torch.zeros(B, 1, Hp, Wp, device=device)

    # key, query, coverage 결합 -> energy map
    def forward(self, query, F2d, cov=None, mask=None, mask_w=None, stride_w=None):
        B, C, Hp, Wp = F2d.shape
        keys   = self.W_F(F2d)                                 # (B, attn_dim, H', W')
        query_ = self.W_h(query).unsqueeze(-1).unsqueeze(-1)   # (B, attn_dim, 1, 1)
        cov_   = self.W_cov(cov) if cov is not None else 0     # (B, attn_dim, H', W') or 0
        e = torch.tanh(keys + query_ + cov_)                   # (B, attn_dim, H', W')
        scores = (e * self.v_att.view(1, -1, 1, 1)).sum(dim=1) # (B, H', W')

        # mask 있으면 사용, 없으면 mask_w + stride_w로 2D mask 생성
        if mask is None and (mask_w is not None and stride_w is not None):
            mask_w = mask_w.to(F2d.device)
            enc_mask_w = F.max_pool1d(
                mask_w.float().unsqueeze(1), kernel_size=stride_w,
                stride=stride_w, ceil_mode=True
            ).squeeze(1).bool()       # (B, W')
            mask = enc_mask_w.unsqueeze(1).expand(-1, Hp, -1)  # (B, H', W')

        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))  # padding 위치 -inf

        # H'*W'에 대한 softmax -> 2D spatial attention
        alpha = F.softmax(scores.view(B, -1), dim=-1).view(B, Hp, Wp)   # (B, H', W')
        # context, coverage 업데이트
        context = (F2d * alpha.unsqueeze(1)).sum(dim=(2, 3))   # (B, C)
        cov_next = (cov if cov is not None else 0) + alpha.unsqueeze(1)   # (B, 1, H', W')
        return context, alpha, cov_next

# DLA

- 이후 추가 모델 개선방안 고민
  - contrastive learning 및 data augmentation (scale augmentation 등)
  - BYOL, DINO 같은 방법 참고해보기? (distilation)
  - DA, DG, SFTA 방법론 참고한 다른 시도
  - multi resolution 기반 실험

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DLAModel(nn.Module):
    # --------- setting ---------
    PAD_ID = 0   # 디코더에서 padding_idx=0으로 고정
    SOS_ID = 1
    EOS_ID = 2
    UNK_ID = 3

    def __init__(self, model_config=None):
        super().__init__()
        model_config = model_config

        # 설정 파라미터
        self.vocab_size = model_config.get("vocab_size", 113)
        self.enc_out_dim = model_config.get("encoder_out_channels", 1024)
        self.dec_emb_dim = model_config.get("decoder_emb_dim", 256)
        self.dec_hid_dim = model_config.get("decoder_hidden_dim", 512)

        # 인코더 (PME, HME 별도 -> 공유 X)
        self.encoder_pme = Encoder()
        self.encoder_hme = Encoder()

        # 디코더 (PME, HME 동일 -> 공유 O)
        # 이 부분 수정함
        self.decoder = Decoder(self.vocab_size, self.dec_emb_dim, self.dec_hid_dim, enc_dim=self.enc_out_dim*4)

    # --------- utils ---------
    @staticmethod
    def _to_seq(enc_out):
        """
        인코더 2D feature map -> 디코더 1D 시퀀스 (column-wise)
        enc_out: (B, C, H', W') 또는 (B, T, D)
        returns: (B, T=W', D=H'*C)
        """
        if enc_out.dim() == 4:   # 2D feature map일 때만 변환
            B, C, Hp, Wp = enc_out.shape
            # column-wise: 각 가로 column(W')마다 H' x C를 펴서 특징으로
            seq = enc_out.permute(0, 3, 2, 1).contiguous().view(B, Wp, Hp * C)
            return seq
        elif enc_out.dim() == 3:
            return enc_out  # 이미 (B,T,D)면 그대로 반환
        else:
            raise ValueError(f"Unexpected encoder output dim: {enc_out.shape}")

    @staticmethod
    def _make_enc_mask(mask_w, T_enc, device):
        """
        원본 폭 마스크를 인코더 시퀀스 길이에 맞게 축약 -> 디코더 attn에서 패딩 위치 가림
        mask_w: (B, W) 1D 폭 마스크 (True=유효) 또는 None
        T_enc:  인코더 시퀀스 길이(W')
        returns: (B, T_enc) bool (각 시퀀스 타임스텝 유효한지) or None
        """
        if mask_w is None:
            return None
        # adaptive pooling으로 W -> T_enc에 맞춰 축약 (윈도우 내 하나라도 True면 True)
        enc_mask = F.adaptive_max_pool1d(mask_w.float().unsqueeze(1).to(device), output_size=T_enc)
        return enc_mask.squeeze(1).bool()   # bool 마스크로 복원

    def _decode_branch(self, enc_seq, tgt=None, enc_mask=None, teacher_forcing_ratio=0.5, max_len=None):
        """
        인코더 시퀀스 + 정답 토큰 시퀀스 -> attn 디코더 1 step 실행
        teacher forcing, EOS all 포함
        enc_seq : (B, T_enc, D_enc) -> _to_seq의 결과
        tgt     : (B, T_out) or None -> 정답 토큰 (학습) or None (추론)
        enc_mask: (B, T_enc) bool or None -> 패딩 가리기용 마스크
        teacher_forcing_ratio: 각 step에서 정답 넣을 확률
        max_len: 추론 시 최대 길이 (기본 128)
        """
        device = enc_seq.device
        B, T_enc, _ = enc_seq.shape

        # 디코더 hidden 초기화
        num_layers = self.decoder.gru.num_layers
        hidden = torch.zeros(num_layers, B, self.dec_hid_dim, device=device)    # GRU hidden 초기화

        # 디코딩 길이 결정
        if tgt is not None:
            T_out = tgt.size(1)   # 학습
        else:
            T_out = max_len if max_len is not None else 128   # 추론

        prev_token = torch.full((B,), self.SOS_ID, dtype=torch.long, device=device)   # SOS_ID로 시작
        logits_list = []

        # 반복 루프
        """
        logits: (B, V) -> 다음 토큰 분포
        hidden: 다음 시점 상태
        context_vec: attn context vector (B, D_ctx)
        attn_w: attn weight (B, T_enc)
        """
        for t in range(T_out):
            logits, hidden, context_vec, attn_w = self.decoder(prev_token, hidden, enc_seq, enc_mask)
            logits_list.append(logits.unsqueeze(1))  # (B, 1, V)

            # 입력 토큰 갱신 (학습이면 정답 토큰 or 모델 예측, 추론이면 항상 모델 예측)
            if (tgt is not None) and (torch.rand(1).item() < teacher_forcing_ratio):
                prev_token = tgt[:, t]   # 정답 토큰
            else:
                prev_token = logits.argmax(dim=-1)   # 모델 예측

            # 추론에서 모두 EOS면 조기 종료
            if tgt is None:
                if torch.all(prev_token == self.EOS_ID):
                    break

        return torch.cat(logits_list, dim=1)  # (B, T', V)

    # --------- forward ---------
    def forward(self, images_pme, images_hme=None, tgt_pme=None, tgt_hme=None,
                mask_w_pme=None, mask_w_hme=None, teacher_forcing_ratio=0.5, max_len=None):
        """
        images_pme/hme : (B, 1, 128, W')
        tgt_pme/hme    : (B, T_out) 또는 None (있으면 학습, 없으면 추론)
        mask_w_pme/hme : (B, W) bool (가로 패딩 마스크). 없으면 None
        returns        : logits_pme -> (B, T_p, V), logits_hme -> (B, T_h, V) or None
        """
        # --- PME ---
        enc_out_pme = self.encoder_pme(images_pme)           # (B, C, H', W') or (B, T, D)
        enc_seq_pme = self._to_seq(enc_out_pme)              # (B, T_enc, D_enc)
        enc_mask_pme = self._make_enc_mask(mask_w_pme, enc_seq_pme.size(1), images_pme.device)

        """
        시작 토큰 <sos>
        학습: teacher forcing or 예측 -> 길이 T_out = tgt.size(1)
        추론: max_len까지 예측 진행, 배치 전체 EOS 나오면 조기 종료
        returns: (B, T_p, V)
        """
        logits_pme = self._decode_branch(
            enc_seq_pme, tgt=tgt_pme, enc_mask=enc_mask_pme,
            teacher_forcing_ratio=teacher_forcing_ratio, max_len=max_len
        )

        # --- HME (paired일 때만) ---
        logits_hme = None
        if images_hme is not None:
            enc_out_hme = self.encoder_hme(images_hme)
            enc_seq_hme = self._to_seq(enc_out_hme)
            enc_mask_hme = self._make_enc_mask(mask_w_hme, enc_seq_hme.size(1), images_hme.device)

            logits_hme = self._decode_branch(
                enc_seq_hme, tgt=tgt_hme, enc_mask=enc_mask_hme,
                teacher_forcing_ratio=teacher_forcing_ratio, max_len=max_len
            )

        return logits_pme, logits_hme

# Dual Loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DualLoss(nn.Module):
    """
    Loss = LD(Xh) + LD(Xp) + LD(X̄) + λ * Lmatch(Xh, Xp)
    CE 정렬: 타깃 [SOS, ..., EOS] 형태 가정 -> logits[:, :-1, :]와 targets[:, 1:]에 맞춰 학습
    Matching Loss: PAD/길이 마스크 적용 -> 유효 토큰 구간에서만 MSE 계산
    """
    def __init__(self, match_weight=0.2, ignore_index=0):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.mse_loss = nn.MSELoss(reduction="mean")   # 실제 계산은 mask로 직접 평균
        self.match_weight = match_weight
        self.ignore_index = ignore_index

    def _ce_aligned(self, logits, targets):
        """
        logits:  (B, T, V)
        targets: (B, T)  # [SOS, ..., EOS] 가정
        returns: scalar CE loss
        """
        B, T, V = logits.size()
        # [SOS, y0, ..., EOS] 기준 정렬 → (T-1) 타임스텝 사용
        logits_aligned  = logits[:, :-1, :]   # 예측 시점 0..T-2
        targets_aligned = targets[:, 1:]      # 정답 시점 1..T-1

        return self.ce_loss(
            logits_aligned.reshape(-1, V),
            targets_aligned.reshape(-1)
        ), logits_aligned.size(1)  # (loss, T')

    def _masked_mse(self, ctx_h, ctx_p, mask=None):
        """
        ctx_* : (B, T, C)
        mask  : (B, T) bool (True=유효). None이면 전체 사용
        returns: scalar masked MSE
        """
        B, T, C = ctx_h.size()
        diff2 = (ctx_h - ctx_p) ** 2  # (B, T, C)

        if mask is None:
            return diff2.mean()

        # 유효 위치가 하나도 없을 때 0으로
        if not mask.any():
            return torch.zeros((), device=ctx_h.device, dtype=ctx_h.dtype)

        # mask를 채널 차원까지 확장
        mask_exp = mask.unsqueeze(-1)  # (B, T, 1)
        # 유효 위치의 평균
        return diff2.masked_select(mask_exp).mean()

    def forward(self,
                logits_h=None, targets_h=None,    # (B, T, V), (B, T)
                logits_p=None, targets_p=None,    # (B, T, V), (B, T)
                logits_up=None, targets_up=None,  # (B, T, V), (B, T)
                context_h=None, context_p=None    # (B, T, C), (B, T, C)
               ):
        """
        logits_*: unnormalized 디코더 출력 (B, T, V)
        targets_*: target indices (B, T)
        context_*: attn context vectors (B, T, C)
        """
        # device 일관성, 초기화
        device = None
        for t in (logits_h, targets_h, logits_p, targets_p, logits_up, targets_up, context_h, context_p):
            if isinstance(t, torch.Tensor):
                device = t.device
                break
        if device is None:
            device = torch.device("cpu")

        loss_total = torch.zeros((), device=device)
        loss_h     = torch.zeros((), device=device)
        loss_p     = torch.zeros((), device=device)
        loss_up    = torch.zeros((), device=device)
        match_loss = torch.zeros((), device=device)

        # Decoder Loss - Handwritten
        T_h_aligned = None
        if logits_h is not None and targets_h is not None:
            loss_h, T_h_aligned = self._ce_aligned(logits_h, targets_h)
            loss_total = loss_total + loss_h

        # Decoder Loss - Paired Printed
        T_p_aligned = None
        if logits_p is not None and targets_p is not None:
            loss_p, T_p_aligned = self._ce_aligned(logits_p, targets_p)
            loss_total = loss_total + loss_p

        # Decoder Loss - Unpaired PME
        if logits_up is not None and targets_up is not None:
            loss_up, _ = self._ce_aligned(logits_up, targets_up)
            loss_total = loss_total + loss_up

        # Context Matching Loss -> HME/PME 디코더 context vector 같은 시점끼리 MSE 맞추기, PAD 제외
        if (context_h is not None) and (context_p is not None):
            # 시간 길이 맞추기
            Tm = min(context_h.size(1), context_p.size(1))
            ctx_h = context_h[:, :Tm, :]
            ctx_p = context_p[:, :Tm, :]

            # 가능한 경우, 타깃에서 PAD 마스크를 만들어 유효 토큰만 매칭
            mask = None
            if (targets_h is not None) and (targets_p is not None):
                # CE에서 [:-1] / [1:] 정렬 -> 타깃도 동일하게 정렬 후 길이 맞추기
                tgt_h_aligned = targets_h[:, 1:] if targets_h.size(1) > 1 else targets_h[:, :0]
                tgt_p_aligned = targets_p[:, 1:] if targets_p.size(1) > 1 else targets_p[:, :0]

                # ctx 길이와 맞추기 (보통 ctx 길이 = logits 길이, CE 정렬 후 T-1)
                Th = min(tgt_h_aligned.size(1), Tm)
                Tp = min(tgt_p_aligned.size(1), Tm)
                Tm2 = min(Th, Tp)   # 공통 유효 길이

                if Tm2 > 0:
                    ctx_h = ctx_h[:, :Tm2, :]
                    ctx_p = ctx_p[:, :Tm2, :]
                    mask_h = (tgt_h_aligned[:, :Tm2] != self.ignore_index)   # PAD 아닌 위치 True
                    mask_p = (tgt_p_aligned[:, :Tm2] != self.ignore_index)   # PAD 아닌 위치 True
                    mask = mask_h & mask_p  # 공통 유효 위치만 True
                else:
                    # 유효 구간이 전혀 없으면 mask=None으로 두고 평균 0 처리 -> 타깃이 없으면 어디가 PAD인지 모름
                    mask = torch.zeros((ctx_h.size(0), ctx_h.size(1)), dtype=torch.bool, device=device)

            match_loss = self._masked_mse(ctx_h, ctx_p, mask)
            loss_total = loss_total + (self.match_weight * match_loss)

        stats = {
            "loss_h":     loss_h.detach().item(),
            "loss_p":     loss_p.detach().item(),
            "loss_up":    loss_up.detach().item(),
            "match_loss": match_loss.detach().item(),
            "total":      loss_total.detach().item(),
        }
        return loss_total, stats

# Metrics

In [ ]:
import torch
import numpy as np
from rapidfuzz.distance import Levenshtein as RLev

def _to_list_batch(x):
    """(B,T) torch.Tensor / np.ndarray / list[list[int]] -> list[list[int]]"""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().tolist()
    if isinstance(x, np.ndarray):
        return x.tolist()
    return x

def strip_special(seq, pad_id=0, sos_id=None, eos_id=None):
    """
    PAD 제거, 맨 앞 SOS 제거, 첫 EOS 이전까지만 사용
    """
    out = [t for t in seq if t != pad_id]
    if sos_id is not None and len(out) and out[0] == sos_id:
        out = out[1:]
    if eos_id is not None and eos_id in out:
        out = out[:out.index(eos_id)]
    return out

def levenshtein_distance(seq1, seq2):
    """
    기본적인 편집 거리 계산 함수 (DP 기반)
    """
    return RLev.distance(seq1, seq2)

def exprate_k(preds, targets, pad_id=0, sos_id=None, eos_id=None):
    """
    Expression rate-k: 예측 수식과 정답 수식이 k개 이하의 토큰 차이만 있을 때 정답으로 간주
    """
    preds = _to_list_batch(preds)
    targets = _to_list_batch(targets)
    assert len(preds) == len(targets)
    N = len(preds)
    cnt = 0
    for p, t in zip(preds, targets):
        pred = strip_special(p, pad_id, bos_id, eos_id)
        target = strip_special(t, pad_id, bos_id, eos_id)
        s, i, d = edit_ops(target, pred)
        if (s + i + d) <= k:
            cnt += 1
    return (cnt / N) if N > 0 else 0.0

def wer(preds, targets, pad_id=0, sos_id=None, eos_id=None):
    """
    WER = (N_sub + N_del + N_ins) / N_Y
    (토큰 단위. 특수토큰 제외한 정답 토큰 수 N_Y 기준)
    """
    D_sum, N_Y = 0, 0
    for p, t in zip(preds, targets):
        ref = strip_special(t, pad_id, sos_id, eos_id)  # target
        hyp = strip_special(p, pad_id, sos_id, eos_id)  # prediction
        D_sum += RLev.distance(ref, hyp)
        N_Y   += len(ref)
    return (D_sum / N_Y) if N_Y > 0 else 0.0

# Utils

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed: int = 42, deterministic: bool = True):
    """재현 가능한 실험을 위한 시드 고정."""
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [ ]:
import torch

def _pick_device(pref="auto"):
  if pref == "cuda":
    return "cuda" if torch.cuda.is_available() else "cpu"
  if pref == "mps":
    if torch.cuda.is_available():
        return "cuda"
    return "mps" if getattr(torch.backends,"mps",None) and torch.backends.mps.is_available() else "cpu"
  if pref == "cpu":
    return "cpu"
  return "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends,"mps",None) and torch.backends.mps.is_available() else "cpu")

In [ ]:
import os
import json

def finalize_config(cfg: dict):
    if "paths" in cfg and all(k in cfg["paths"] for k in ("run_dir","ckpt_dir","log_dir")):
        for k in ("ckpt_dir","log_dir"):
            os.makedirs(cfg["paths"][k], exist_ok=True)
        return

    cfg["misc"]["device"] = _pick_device(cfg["misc"].get("device","auto"))
    run_id = f'{cfg["experiment_name"]}-{datetime.now().strftime("%y%m%d-%H%M%S")}'
    base = os.path.join("runs", run_id)
    paths = {
      "run_dir": base,
      "ckpt_dir": os.path.join(base, "checkpoints"),
      "log_dir": os.path.join(base, "logs"),
      "config_json": os.path.join(base, "config.json"),
      "batch_log_csv": os.path.join(base, "logs", "batch_log.csv"),
      "best_ckpt": os.path.join(base, "checkpoints", "best.pth"),
      "last_ckpt": os.path.join(base, "checkpoints", "last.pth"),
    }
    os.makedirs(paths["ckpt_dir"], exist_ok=True)
    os.makedirs(paths["log_dir"], exist_ok=True)
    cfg["paths"] = paths
    with open(paths["config_json"], "w", encoding="utf-8") as f:
      json.dump(cfg, f, ensure_ascii=False, indent=2)

In [ ]:
import os
import json

def save_log(log_dict, save_path="log.json"):
    """학습 로그를 JSON으로 저장."""
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(log_dict, f, ensure_ascii=False, indent=2)
    print(f"로그 저장 완료: {save_path}")

In [ ]:
import matplotlib.pyplot as plt
import torch

def plot_loss_curve(loss_list, save_path=None):
    """Loss 곡선 시각화."""
    plt.figure(figsize=(8, 4))
    plt.plot(loss_list, marker='o', label="Train Loss")
    plt.title("Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path)
        print(f"Loss curve saved to {save_path}")
    else:
        plt.show()

def visualize_prediction(image, pred_str, target_str, figsize=(10, 3)):
    """수식 이미지와 예측/정답 문자열 시각화"""
    if isinstance(image, torch.Tensor):
        image = image.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=figsize)
    plt.imshow(image.squeeze(), cmap='gray')
    plt.title(f"Pred: {pred_str}\nTarget: {target_str}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

def plot_learning_curve(losses, title="Training Loss"):
    plt.plot(losses, label="loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.grid()
    plt.legend()
    plt.show()


def token_accuracy(preds, targets, ignore_idx=0):
    T = min(preds.size(1), targets.size(1))
    preds = preds[:, :T]
    targets = targets[:, :T]
    mask = targets.ne(ignore_idx)
    correct = (preds.eq(targets) & mask).sum().item()
    total = mask.sum().item()
    return (correct / total) if total > 0 else 0.0

# Config

In [ ]:
# import os, json
# from datetime import datetime
# import torch

# CFG = {
#   "experiment_name": "dla_baseline",

#   "data": {
#     "vocab": "data/vocab/crohme_vocab.txt",
#     "paired": {
#       "hme_img": "data/preprocessed/train/paired/crohme/hme_preprocessed",
#       "pme_img": "data/preprocessed/train/paired/crohme/pme_preprocessed",
#       "caption": "data/preprocessed/train/paired/crohme/caption.txt",
#     },
#     "unpaired": {
#       "pme_img": "data/preprocessed/train/paired/im2latex/pme_cropped",
#       "caption": "data/preprocessed/train/paired/im2latex/caption.txt",
#     },
#   },

#   "model": {
#     "vocab_size": 113,
#     "encoder_out_channels": 1024,
#     "decoder_emb_dim": 256,
#     "decoder_hidden_dim": 512,
#     "attention_dim": 1024,
#     "attn_out_dim": 1024,
#   },

#   "training": {
#     "batch_size": 1,
#     "epochs": 100,
#     "learning_rate": 0.1,
#     "match_weight": 0.2,
#     "optimizer": "adadelta",
#     "grad_clip": 5.0,
#     "early_stop_patience": 5,
#     "ignore_idx": 0,
#     "scheduler": {"use": True, "type": "StepLR", "step_size": 10, "gamma": 0.5},
#   },

#   "testing": {"batch_size": 1, "max_len": 150},

#   "misc": {"seed": 42, "device": "auto"}
# }

# Train

In [ ]:
# import os, json, random
# from pathlib import Path
# from datetime import datetime

# import torch
# from torch.utils.data import DataLoader
# from tqdm import tqdm
# import matplotlib.pyplot as plt

# assert "CFG" in globals(), "CFG dict 먼저 정의"

# # 디바이스, 시드, 경로 준비
# CFG["misc"]["device"] = pick_device(CFG["misc"].get("device", "auto"))
# DEVICE = torch.device(CFG["misc"]["device"])
# set_seed(CFG["misc"]["seed"], deterministic=True)
# finalize_paths(CFG)

# print("Using device:", DEVICE)
# print("Run dir:", CFG["paths"]["run_dir"])

# # Vocab
# vocab = Vocab.load_from_txt(Path(CFG["data"]["vocab"]))
# vocab_size = len(vocab)

# # Transforms
# trans_cfg = CFG.get("transforms", None)
# try:
#     hme_paired_transform = get_formula_transform("paired_hme", trans_cfg)
#     pme_paired_transform = get_formula_transform("paired_pme", trans_cfg)
#     unpaired_transform   = get_formula_transform("unpaired",   trans_cfg)
# except Exception:
#     hme_paired_transform = pme_paired_transform = unpaired_transform = None

# # Dataset
# paired_dataset = PairedFormulaDataset(
#     hme_dir=Path(CFG["data"]["paired"]["hme_img"]),
#     pme_dir=Path(CFG["data"]["paired"]["pme_img"]),
#     caption_path=Path(CFG["data"]["paired"]["caption"]),
#     transform_hme=hme_paired_transform,
#     transform_pme=pme_paired_transform,
#     vocab=vocab,
# )
# unpaired_dataset = FormulaDataset(
#     image_dir=Path(CFG["data"]["paired"]["pme_img"]),
#     caption_path=Path(CFG["data"]["paired"]["caption"]),
#     transform=unpaired_transform,
#     vocab=vocab,
# )

# # Dataloader
# pad_idx = CFG["training"]["ignore_idx"]
# num_workers = CFG.get("num_workers", 2)
# BATCH_SIZE = CFG["training"]["batch_size"]

# collate_fn_1 = lambda b: formula_collate_fn(b, pad_idx=pad_idx)
# collate_fn_2 = lambda b: paired_collate_fn(b, pad_idx=pad_idx)

# unpaired_loader = DataLoader(
#     unpaired_dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_fn_1,
#     num_workers=num_workers, pin_memory=True,
# )
# paired_loader = DataLoader(
#     paired_dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_fn_2,
#     num_workers=num_workers, pin_memory=True,
# )

# # Model, Loss, Optimizer, Scheduler
# model = DLAModel(vocab_size=vocab_size, model_config=CFG["model"]).to(DEVICE)
# criterion = DualLoss(match_weight=CFG["training"]["match_weight"], ignore_index=pad_idx)

# opt_name = CFG["training"].get("optimizer", "adadelta").lower()
# params = model.parameters()
# if opt_name == "adam":
#     optimizer = torch.optim.Adam(params, lr=CFG["training"]["learning_rate"])
# elif opt_name == "sgd":
#     optimizer = torch.optim.SGD(params, lr=CFG["training"]["learning_rate"], momentum=0.9)
# elif opt_name == "adadelta":
#     optimizer = torch.optim.Adadelta(params, lr=CFG["training"]["learning_rate"])
# else:
#     raise ValueError(f"Unknown optimizer: {opt_name}")

# scheduler = None
# sch_cfg = CFG["training"].get("scheduler", {"use": False})
# if sch_cfg.get("use", False):
#     if sch_cfg["type"] == "StepLR":
#         scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=sch_cfg["step_size"], gamma=sch_cfg["gamma"])
#     else:
#         raise NotImplementedError(f"Unsupported scheduler: {sch_cfg['type']}")

# # Train loop
# EPOCHS = CFG["training"]["epochs"]
# GRAD_CLIP = CFG["training"]["grad_clip"]
# best_loss = float("inf")
# patience = 0
# loss_history = []
# log_dict = {"train": []}

# batch_log_dir = Path(CFG["paths"]["batch_log_dir"])
# save_best = Path(CFG["paths"]["best_ckpt"])
# save_last = Path(CFG["paths"]["last_ckpt"])

# unpaired_iter = iter(unpaired_loader)

# for epoch in range(1, EPOCHS + 1):
#     model.train()
#     total_loss, total_acc, num_batches = 0.0, 0.0, 0

#     loop = tqdm(paired_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True)
#     for batch_pair in loop:
#         try:
#             batch_up = next(unpaired_iter)
#         except StopIteration:
#             unpaired_iter = iter(unpaired_loader)
#             batch_up = next(unpaired_iter)

#         img_pme = batch_pair["img_pme"].to(DEVICE, non_blocking=True)
#         img_hme = batch_pair["img_hme"].to(DEVICE, non_blocking=True)
#         tgt     = batch_pair["formula"].to(DEVICE, non_blocking=True)
#         img_up  = batch_up["image"].to(DEVICE, non_blocking=True)
#         tgt_up  = batch_up["formula"].to(DEVICE, non_blocking=True)

#         # Forward (teacher forcing)
#         logits_p, logits_h, context_p, context_h = model(img_pme, img_hme, tgt, tgt)
#         logits_up, _, _, _ = model(img_up, None, tgt_up, None)

#         loss, loss_dict = criterion(
#             logits_h, tgt,           # HME
#             logits_p, tgt,           # PME
#             logits_up, tgt_up,       # Unpaired PME
#             context_h, context_p,    # Context matching
#         )

#         optimizer.zero_grad(set_to_none=True)
#         loss.backward()
#         if GRAD_CLIP and GRAD_CLIP > 0:
#             torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
#         optimizer.step()

#         with torch.no_grad():
#             preds = logits_h.argmax(dim=-1)
#             acc = token_accuracy(preds, tgt[:, 1:], ignore_idx=pad_idx)  # SOS 제외 가정

#         total_loss += float(loss.item())
#         total_acc  += float(acc)
#         num_batches += 1

#         loop.set_postfix(loss=loss.item(), acc=acc)

#     avg_loss = total_loss / num_batches
#     avg_acc  = total_acc  / num_batches
#     print(f"\n[Epoch {epoch}] Loss: {avg_loss:.4f}, Token Acc: {avg_acc:.4f}")

#     log_dict["train"].append({"epoch": epoch, "loss": avg_loss, "token_accuracy": avg_acc})
#     loss_history.append(avg_loss)

#     if scheduler:
#         scheduler.step()

#     # 마지막 체크포인트
#     torch.save({"model": model.state_dict()}, save_last)

#     # 최고 성능 저장, 조기종료
#     if epoch == 1 or avg_loss < best_loss:
#         best_loss = avg_loss
#         patience = 0
#         torch.save({"model": model.state_dict()}, save_best)
#         print("Best model saved.")
#     else:
#         patience += 1
#         if patience >= CFG["training"]["early_stop_patience"]:
#             print(f"Early stopping at epoch {epoch}")
#             break

# # 결과 저장
# save_log(log_dict, CFG["paths"]["train_log_json"])
# plot_loss_curve(loss_history, CFG["paths"]["loss_curve_png"])
# print("Best loss:", best_loss, "| Best ckpt:", CFG["paths"]["best_ckpt"])

# Test

In [ ]:
# from pathlib import Path
# from datetime import datetime
# import json
# from tqdm import tqdm

# import torch
# from torch.utils.data import DataLoader

# assert "CFG" in globals(), "CFG dict 먼저 정의"

# DEVICE = torch.device(_pick_device(CFG["misc"].get("device", "auto")))
# set_seed(CFG["misc"]["seed"])
# print("Using device:", DEVICE)

# # 결과 출력용 디렉터리(run_id 별도 생성)
# test_run_id = f'{CFG["experiment_name"]}-test-{datetime.now().strftime("%y%m%d-%H%M%S")}'
# OUT_DIR   = Path("runs") / test_run_id
# LOG_DIR   = OUT_DIR / "logs"
# PREDS_DIR = OUT_DIR / "preds"
# for p in (LOG_DIR, PREDS_DIR):
#     p.mkdir(parents=True, exist_ok=True)
# RESULTS_JSON = LOG_DIR / "test_results.json"

# # 체크포인트 경로 결정, 로드
# ckpt_path = CFG["misc"].get("checkpoint_path", None)
# if ckpt_path:
#     ckpt_path = Path(ckpt_path)
# elif "paths" in CFG and CFG["paths"].get("best_ckpt"):
#     ckpt_path = Path(CFG["paths"]["best_ckpt"])
# else:
#     raise FileNotFoundError("Checkpoint not found.")
# if not ckpt_path.exists():
#     raise FileNotFoundError(f"No checkpoint exists: {ckpt_path}")

# # Vocab, Model
# vocab = Vocab.load_from_txt(Path(CFG["data"]["vocab"]))
# SOS_ID = vocab.token2idx["<sos>"]
# EOS_ID = vocab.token2idx["<eos>"]

# model = DLAModel(vocab_size=len(vocab), model_config=CFG["model"]).to(DEVICE)
# state = torch.load(ckpt_path, map_location=DEVICE)
# model.load_state_dict(state["model"])
# model.eval()

# # 평가 함수
# BATCH_SIZE = CFG["testing"].get("batch_size", 1)
# MAX_LEN    = CFG["testing"].get("max_len", 150)
# IGNORE_IDX = CFG["training"]["ignore_idx"]
# NUM_WORKERS = CFG.get("num_workers", 2)

# def evaluate(img_dir: Path, caption_path: Path, mode: str, tag: str):
#     """mode in {'hme','pme'}; tag는 출력 파일명에 사용."""
#     assert mode in {"hme", "pme"}
#     trans_key = "paired_hme" if mode == "hme" else "paired_pme"
#     transform = get_formula_transform(trans_key, CFG.get("transforms", None))

#     dataset = FormulaDataset(image_dir=img_dir, caption_path=caption_path, transform=transform, vocab=vocab)
#     loader = DataLoader(
#         dataset,
#         batch_size=BATCH_SIZE,
#         shuffle=False,
#         num_workers=NUM_WORKERS,
#         pin_memory=True,
#         collate_fn=lambda b: formula_collate_fn(b, pad_idx=IGNORE_IDX),
#     )

#     preds, targets, debug_lines = [], [], []
#     with torch.no_grad():
#         for batch in tqdm(loader, desc=f"[{tag}] {mode.upper()}"):
#             imgs    = batch["image"].to(DEVICE, non_blocking=True)
#             tgt_ids = batch["formula"]

#             pred_ids = model.predict(imgs, max_len=MAX_LEN, sos_idx=SOS_ID, eos_idx=EOS_ID)
#             pred_tokens   = decode_sequence(pred_ids, vocab)
#             target_tokens = decode_sequence(tgt_ids, vocab)

#             preds.extend(pred_tokens)
#             targets.extend(target_tokens)

#             for pr, gt in zip(pred_tokens, target_tokens):
#                 debug_lines.append(f"[GT]   {' '.join(gt)}")
#                 debug_lines.append(f"[PRD]  {' '.join(pr)}")
#                 debug_lines.append("---")

#     debug_path = PREDS_DIR / f"pred_target_pairs_{mode}_{tag}.txt"
#     with open(debug_path, "w", encoding="utf-8") as f:
#         f.write("\n".join(debug_lines))
#     print(f"{mode.upper()} {tag} 결과 저장: {debug_path}")
#     return preds, targets

# def metrics_dict(preds, targets):
#     out = {f"exprate_{k}": round(exprate_k(preds, targets, k), 4) for k in range(4)}
#     out["cer"] = round(cer(preds, targets), 4)
#     return out

# def print_section(title: str, d: dict):
#     print(f"\n{title}")
#     for k, v in d.items():
#         print(f"  {k}: {v}")

# # 전체 평가
# TEST_YEARS = CFG["testing"].get("years", ["2014", "2016", "2019"])
# base = Path(".").resolve()
# result_log = {"CROHME": {"hme": {}, "pme": {}}, "IM2LATEX": {"pme": {}}}

# # HME (CROHME)
# for year in TEST_YEARS:
#     img_dir = base / f"data/preprocessed/test/hme/crohme/{year}/hme_img"
#     cap_path = base / f"data/preprocessed/test/hme/crohme/{year}/caption.txt"
#     preds, tgts = evaluate(img_dir, cap_path, mode="hme", tag=f"crohme_{year}")
#     result_log["CROHME"]["hme"][year] = metrics_dict(preds, tgts)

# # PME (CROHME)
# for year in TEST_YEARS:
#     img_dir = base / f"data/preprocessed/test/pme/crohme/{year}/pme_img"
#     cap_path = base / f"data/preprocessed/test/pme/crohme/{year}/caption.txt"
#     preds, tgts = evaluate(img_dir, cap_path, mode="pme", tag=f"crohme_{year}")
#     result_log["CROHME"]["pme"][year] = metrics_dict(preds, tgts)

# # PME (IM2LATEX)
# img_dir = base / "data/preprocessed/test/pme/im2latex/img"
# cap_path = base / "data/preprocessed/test/pme/im2latex/caption.txt"
# preds, tgts = evaluate(img_dir, cap_path, mode="pme", tag="im2latex")
# result_log["IM2LATEX"]["pme"] = metrics_dict(preds, tgts)

# # 출력, 저장
# for y in TEST_YEARS:
#     print_section(f"CROHME-HME {y}", result_log["CROHME"]["hme"][y])
#     print_section(f"CROHME-PME {y}", result_log["CROHME"]["pme"][y])
# print_section("IM2LATEX-PME", result_log["IM2LATEX"]["pme"])

# with open(RESULTS_JSON, "w", encoding="utf-8") as f:
#     json.dump(result_log, f, ensure_ascii=False, indent=2)

# print(f"\n평가 완료! 결과 JSON: {RESULTS_JSON}")

# _best = f"{best_loss:.4f}" if isinstance(best_loss, (int, float)) else "N/A"
# print(f"Best loss: {_best} | Best ckpt: {ckpt_path}")

In [ ]:
import os
import cv2
import numpy as np
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

def to_tensor(img):
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    x = torch.from_numpy(img.astype(np.float32) / 255.0)[None, None, ...]  # (1,1,128,W)
    return x

def make_dummy_tgt(T=20, vocab_size=113, SOS=1, EOS=2):
    tgt = torch.randint(low=4, high=vocab_size, size=(1, T), dtype=torch.long)
    tgt[:, 0] = SOS
    tgt[:, -1] = EOS
    return tgt

IMAGE_PATH = "dummy/preprocessed/train/paired/crohme/hme_preprocessed/101_Fabricio.png"

# 1) 입력 준비 (리사이즈/유니코드 로더 없음)
img = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
assert img is not None, "이미지 로드 에러"
x = to_tensor(img).to(DEVICE)              # (1,1,128,W)
B, _, H, W = x.shape
mask_w = torch.ones(1, W, dtype=torch.bool, device=DEVICE)

print("=== Input ===")
print(f"Original image shape : {img.shape} (H,W)")
print(f"Input tensor shape   : {tuple(x.shape)}")

# 2) 모델 (세션에 DLAModel이 이미 정의/임포트되어 있다고 가정)
model = DLAModel(model_config={"vocab_size": 112}).to(DEVICE).eval()

# 3) 인코더 출력/시퀀스 확인
with torch.no_grad():
    enc_out = model.encoder_pme(x)         # (1, 1024, 4, W')
print("=== Encoder ===")
print(f"Encoder feature map : {tuple(enc_out.shape)}")

# --- [핵심 패치] 실제 H'로 enc_dim 재설정이 필요한 경우 디코더 재생성 ---
Hp = enc_out.shape[2]                      # 실제 H'
enc_dim_actual = model.enc_out_dim * Hp    # C(=1024) * H'
k_in = model.decoder.attn.k_proj.in_features  # 디코더가 기대하는 enc_dim

if k_in != enc_dim_actual:
    print(f"[fix] Rebuild decoder: enc_dim {k_in} -> {enc_dim_actual} (Hp={Hp})")
    # Decoder, ProjectedDotAttention가 현재 세션에 정의되어 있다고 가정
    model.decoder = Decoder(
        vocab_size=model.vocab_size,
        emb_dim=model.dec_emb_dim,
        hidden_dim=model.dec_hid_dim,
        enc_dim=enc_dim_actual,
        attn_dim=512,              # 디코더에서 사용 중인 값과 일치시킴
    ).to(DEVICE).eval()

enc_seq = model._to_seq(enc_out)           # (1, T_enc=W', D_enc=4*1024)
print(f"Encoder seq shape   : {tuple(enc_seq.shape)}")

# 4A) 학습 스타일(teacher forcing) 테스트
tgt = make_dummy_tgt(T=20, vocab_size=model.vocab_size,
                     SOS=model.SOS_ID, EOS=model.EOS_ID).to(DEVICE)
with torch.no_grad():
    logits_pme_tf, _ = model(
        images_pme=x,
        images_hme=None,
        tgt_pme=tgt,
        tgt_hme=None,
        mask_w_pme=mask_w,
        mask_w_hme=None,
        teacher_forcing_ratio=1.0,  # 항상 정답 입력
    )
print("=== Decoder (Teacher Forcing) ===")
print(f"Logits (train-style): {tuple(logits_pme_tf.shape)}  # (B, T_out, V)")

# 4B) 추론 스타일(그리디) 테스트
with torch.no_grad():
    logits_pme_inf, _ = model(
        images_pme=x,
        images_hme=None,
        tgt_pme=None,               # 추론 모드
        tgt_hme=None,
        mask_w_pme=mask_w,
        mask_w_hme=None,
        teacher_forcing_ratio=0.0,
        max_len=30
    )
preds = logits_pme_inf.argmax(dim=-1)      # (1, T')
print("=== Decoder (Inference) ===")
print(f"Logits (infer-style): {tuple(logits_pme_inf.shape)}  # (B, T', V)")
print(f"Pred ids (first 15) : {preds[0, :15].tolist()}")

=== Input ===
Original image shape : (97, 412) (H,W)
Input tensor shape   : (1, 1, 97, 412)
=== Encoder ===
Encoder feature map : (1, 1024, 3, 12)
[fix] Rebuild decoder: enc_dim 4096 -> 3072 (Hp=3)
Encoder seq shape   : (1, 12, 3072)
=== Decoder (Teacher Forcing) ===
Logits (train-style): (1, 20, 113)  # (B, T_out, V)
=== Decoder (Inference) ===
Logits (infer-style): (1, 30, 113)  # (B, T', V)
Pred ids (first 15) : [94, 16, 103, 103, 4, 67, 82, 102, 86, 28, 35, 1, 105, 97, 65]


In [ ]:
# ==== Pred ids → tokens (using your Vocab) ====
from pathlib import Path

# 1) vocab 불러오기 (토큰 txt가 있을 때)
VOCAB_TXT = Path("dummy/vocab.txt")
vocab = Vocab.load_from_txt(VOCAB_TXT)
assert len(vocab) == model.vocab_size, f"vocab size mismatch: {len(vocab)} vs {model.vocab_size}"

# 인퍼런스 결과 디코드
pred_ids = preds[0].tolist()              # (T',)
pred_text = vocab.decode(pred_ids)        # EOS에서 멈추고 PAD/SOS 제거
print("Pred (decoded):", pred_text)

# (옵션) teacher-forcing 로짓에서 argmax로 디코드 확인도 가능
tf_ids = logits_pme_tf.argmax(dim=-1)[0].tolist()
tf_text = vocab.decode(tf_ids)
print("TF (decoded):", tf_text)


Pred (decoded): > 8 \neq \neq 1 { + \tan \times G Y \forall \geq \pi o 9 \limits E \pi o 9 8 \neq \neq 1 { + \tan \times
TF (decoded): > > \prime - \times F 6 \tan \sin \sin \sin . . C \limits \limits 6 6 > >
